# `usda(series, start, end=None)` and `usda_series(dataset=None, frequency=None)`

These functions give US agriculture data from `indicators.usda`, oldest first. The data comes from the USDA National Agricultural Statistics Service (NASS). It has the yearly US production, yield and price received for corn, soybeans, wheat and cotton, and for cattle, hogs and milk.

`usda_series()` gives the dataset, name, units, frequency and category of each series id. Each cell reads prod as `readonly_user`, and does not write. You can change the arguments and run a cell again.

Use the NASS series id (`CORN, GRAIN - PRICE RECEIVED, MEASURED IN $ / BU`) or the name of a series (`Corn price received`), in any case and spacing. The functions do not give years with no value, and do not fill them.

> **About the data shown.** This product uses the NASS API but is not endorsed or certified by NASS. Source: USDA National Agricultural Statistics Service, Quick Stats, https://quickstats.nass.usda.gov/ (accessed November 27, 2026). arora_data has no affiliation with USDA or NASS. The values are the NASS values, in the NASS units. This notebook is not investment advice.

In [1]:
from datetime import date

import polars as pl

from arora_data.frames import to_df
from arora_data.sync import usda, usda_series
from arora_data.usda import UsdaSeries, UsdaValue


def side_by_side(ids, start):
    """The values of some series, one column for each series, joined on the year."""
    frames = [to_df(usda(i, start), UsdaValue).select(pl.col("date").dt.year().alias("year"), pl.col("value").alias(n))
              for i, n in ids.items()]
    joined = frames[0]
    for f in frames[1:]:
        joined = joined.join(f, on="year", how="full", coalesce=True)
    return joined.sort("year")

## The series
Each call reads the series from the table, thus a new series needs no code change. The labels come from the latest row of each series. `first_date` and `latest_date` are the first and latest years that have a value.

In [2]:
series = to_df(usda_series(), UsdaSeries)
print(series.height, "series")
series.select("name", "series_id", "units", "first_date", "latest_date")

19 series


name,series_id,units,first_date,latest_date
str,str,str,date,date
"""Calves price received""","""CATTLE, CALVES - PRICE RECEIVE…","""$/CWT""",1996-01-01,2025-01-01
"""Cattle production (liveweight)""","""CATTLE, INCL CALVES - PRODUCTI…","""LB""",1988-01-01,2025-01-01
"""Corn price received""","""CORN, GRAIN - PRICE RECEIVED, …","""$/BU""",1996-01-01,2025-01-01
"""Corn production""","""CORN, GRAIN - PRODUCTION, MEAS…","""BU""",1866-01-01,2026-01-01
"""Corn yield""","""CORN, GRAIN - YIELD, MEASURED …","""BU/ACRE""",1866-01-01,2026-01-01
…,…,…,…,…
"""Soybean production""","""SOYBEANS - PRODUCTION, MEASURE…","""BU""",1924-01-01,2026-01-01
"""Soybean yield""","""SOYBEANS - YIELD, MEASURED IN …","""BU/ACRE""",1924-01-01,2026-01-01
"""Wheat price received""","""WHEAT - PRICE RECEIVED, MEASUR…","""$/BU""",1996-01-01,2025-01-01


## One dataset or one frequency
`usda_series(dataset=...)` and `usda_series(frequency=...)` give only the series that have that value. You can use the two filters together. Each commodity is one dataset. All the series are annual now. The number of series for each dataset and frequency:

In [3]:
series.group_by("dataset", "frequency").agg(pl.len().alias("series"), pl.col("first_date").min()).sort("dataset")

dataset,frequency,series,first_date
str,str,u32,date
"""usda/cattle""","""annual""",2,1988-01-01
"""usda/corn""","""annual""",3,1866-01-01
"""usda/cotton""","""annual""",3,1866-01-01
"""usda/hogs""","""annual""",3,1988-01-01
"""usda/milk""","""annual""",2,1924-01-01
"""usda/soybeans""","""annual""",3,1924-01-01
"""usda/wheat""","""annual""",3,1866-01-01


In [4]:
for s in usda_series(dataset="usda/hogs", frequency="annual"):
    print(f"{s.name:<28} {s.units:<6} {s.series_id}")

Hog inventory                HEAD   HOGS - INVENTORY
Hog price received           $/CWT  HOGS - PRICE RECEIVED, MEASURED IN $ / CWT
Hog production (liveweight)  LB     HOGS - PRODUCTION, MEASURED IN LB


## One series, by id or by name
The corn price received for the last 10 years, in dollars per bushel. The id and the name, in any case and spacing, give the same rows. An annual value has the first day of its year. The price is the average of the calendar year, not of the marketing year.

In [5]:
start = date(2016, 1, 1)
assert usda("CORN, GRAIN - PRICE RECEIVED, MEASURED IN $ / BU", start) == usda("corn price received", start) \
    == usda("  Corn   PRICE received ", start)
to_df(usda("Corn price received", start), UsdaValue)

series_id,name,date,value,units
str,str,date,"decimal[38,8]",str
"""CORN, GRAIN - PRICE RECEIVED, …","""Corn price received""",2016-01-01,3.48000000,"""$/BU"""
"""CORN, GRAIN - PRICE RECEIVED, …","""Corn price received""",2017-01-01,3.36000000,"""$/BU"""
"""CORN, GRAIN - PRICE RECEIVED, …","""Corn price received""",2018-01-01,3.47000000,"""$/BU"""
"""CORN, GRAIN - PRICE RECEIVED, …","""Corn price received""",2019-01-01,3.75000000,"""$/BU"""
"""CORN, GRAIN - PRICE RECEIVED, …","""Corn price received""",2020-01-01,3.50000000,"""$/BU"""
"""CORN, GRAIN - PRICE RECEIVED, …","""Corn price received""",2021-01-01,5.40000000,"""$/BU"""
"""CORN, GRAIN - PRICE RECEIVED, …","""Corn price received""",2022-01-01,6.76000000,"""$/BU"""
"""CORN, GRAIN - PRICE RECEIVED, …","""Corn price received""",2023-01-01,5.95000000,"""$/BU"""
"""CORN, GRAIN - PRICE RECEIVED, …","""Corn price received""",2024-01-01,4.26000000,"""$/BU"""


## Crop prices
The price received for corn, soybeans and wheat in dollars per bushel, and for upland cotton in dollars per pound, for the last 10 years. The calendar-year prices start in 1996.

In [6]:
side_by_side({"Corn price received": "corn $/BU", "Soybean price received": "soybeans $/BU",
              "Wheat price received": "wheat $/BU", "Upland cotton price received": "cotton $/LB"}, start)

year,corn $/BU,soybeans $/BU,wheat $/BU,cotton $/LB
i32,"decimal[38,8]","decimal[38,8]","decimal[38,8]","decimal[38,8]"
2016,3.48000000,9.39000000,4.11000000,0.64800000
2017,3.36000000,9.39000000,4.44000000,0.68200000
2018,3.47000000,9.15000000,5.14000000,0.71100000
2019,3.75000000,8.43000000,4.74000000,0.65100000
2020,3.50000000,8.98000000,4.86000000,0.59200000
2021,5.40000000,13.00000000,6.84000000,0.76200000
2022,6.76000000,14.90000000,9.30000000,0.93500000
2023,5.95000000,14.10000000,7.67000000,0.81200000
2024,4.26000000,11.10000000,5.80000000,0.73500000


## Livestock and dairy prices
The price received for hogs, calves and milk, in dollars per hundredweight (100 pounds), for the last 10 years. The hog price starts in 1988, and the other two in 1996.

In [7]:
side_by_side({"Hog price received": "hogs", "Calves price received": "calves", "Milk price received": "milk"}, start)

year,hogs,calves,milk
i32,"decimal[38,8]","decimal[38,8]","decimal[38,8]"
2016,49.60000000,163.00000000,16.30000000
2017,53.60000000,165.00000000,17.70000000
2018,49.90000000,168.00000000,16.30000000
2019,51.90000000,157.00000000,18.70000000
2020,47.10000000,157.00000000,18.10000000
2021,68.50000000,169.00000000,18.50000000
2022,73.00000000,192.00000000,25.30000000
2023,62.00000000,258.00000000,20.30000000
2024,64.10000000,316.00000000,22.60000000


## Units
The values are in the NASS units. "BU" is bushels, "CWT" is hundredweight (100 pounds), and "480 LB BALES" is cotton bales of 480 pounds. The units of the series:

In [8]:
series.group_by("units").agg(pl.col("name").sort().str.join(", ").alias("series")).sort("units")

units,series
str,str
"""$/BU""","""Corn price received, Soybean p…"
"""$/CWT""","""Calves price received, Hog pri…"
"""$/LB""","""Upland cotton price received"""
"""480 LB BALES""","""Cotton production"""
"""BU""","""Corn production, Soybean produ…"
"""BU/ACRE""","""Corn yield, Soybean yield, Whe…"
"""HEAD""","""Hog inventory"""
"""LB""","""Cattle production (liveweight)…"
"""LB/ACRE""","""Cotton yield"""


## Corn yield, by decade
The corn yield series starts in 1866. The average yield of each decade, in bushels per acre:

In [9]:
corn = to_df(usda("Corn yield", date(1866, 1, 1)), UsdaValue)
corn.group_by((pl.col("date").dt.year() // 10 * 10).alias("decade")) \
    .agg(pl.col("value").mean().round(1).alias("BU/ACRE"), pl.len().alias("years")).sort("decade")

decade,BU/ACRE,years
i32,f64,u32
1860,24.2,4
1870,26.6,10
1880,25.9,10
1890,25.9,10
1900,27.3,10
…,…,…
1980,105.9,10
1990,123.3,10
2000,147.2,10


## Hog inventory
`HOGS - INVENTORY` has only 2 values, 2024 and 2025, in head. NASS gives the calendar-year form of this series only for those years. The usual NASS hog inventory is a count on a given day (for example 1 December), and the table does not hold that form now.

In [10]:
to_df(usda("Hog inventory", date(1866, 1, 1)), UsdaValue)

series_id,name,date,value,units
str,str,date,"decimal[38,8]",str
"""HOGS - INVENTORY""","""Hog inventory""",2024-01-01,75075400.00000000,"""HEAD"""
"""HOGS - INVENTORY""","""Hog inventory""",2025-01-01,75414700.00000000,"""HEAD"""


## Revisions
The values of the current year are NASS estimates. NASS revises them, and the main repo then writes over the earlier values. The table does not keep the earlier values.

## Not here
The table does not hold the monthly prices, the marketing-year prices, the grain stocks or the acreage now. `commodities()` gives the futures prices.

## Bad input
An unknown id or name, a filter value that no series has, or a start after the end, raises a clear error before the function reads values.

In [11]:
for call in [lambda: usda("Corn", date(2020, 1, 1)), lambda: usda("Corn price", date(2020, 1, 1)),
             lambda: usda("Corn yield", date(2020, 1, 2), date(2020, 1, 1)),
             lambda: usda_series(frequency="monthly"), lambda: usda_series(dataset="usda/barley")]:
    try:
        call()
    except ValueError as e:
        print(e)

'Corn' is not a stored USDA series: use an id or name from usda_series(), such as 'Corn price received'


'Corn price' is not a stored USDA series: use an id or name from usda_series(), such as 'Corn price received'
start must be on or before end


'monthly' is not a stored USDA frequency: use one of annual


'usda/barley' is not a stored USDA dataset: use one of usda/cattle, usda/corn, usda/cotton, usda/hogs, usda/milk, usda/soybeans, usda/wheat
